# 日频数据准备：VWAP、清洗、mask 与矩阵化

请在三份原始数据下载完成后运行。本 Notebook 调用 `factor_gfn.data.preprocess`，生成 `data/processed/` 下的清洗长表、六特征张量、两个 mask、日期/股票索引和元数据。它不会修改 `data/raw/` 或 `data/download_parts/`。

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from factor_gfn.data.preprocess import PreprocessConfig, inspect_inputs, run_preprocess

config = PreprocessConfig(output_dir=project_root / 'data' / 'processed')
print('原始后复权行情:', config.market_path)
print('原始不复权收盘:', config.raw_close_path)
print('处理结果目录:', config.output_dir)

In [ ]:
input_summary = inspect_inputs(config)
input_summary

# 请人工确认：两份行情的股票数、日期范围和下载日志均符合预期后，再运行下一格。

In [ ]:
RUN_PREPROCESS = True  # 下载完成并检查上方摘要后，手动改为 True
assert RUN_PREPROCESS, '尚未确认执行；请先完成下载检查，再将 RUN_PREPROCESS 改为 True'

# 若 Windows 提示目标文件被占用，请关闭读取 data/processed 文件的变量或程序后重试。
metadata = run_preprocess(config)
metadata

In [ ]:
required = [
    config.daily_clean_path, config.data_tensor_path, config.valid_mask_path,
    config.universe_mask_path, config.date_list_path, config.stock_list_path,
    config.metadata_path,
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'缺少处理结果：{missing}'

tensor = np.load(config.data_tensor_path, mmap_mode='r')
valid_mask = np.load(config.valid_mask_path, mmap_mode='r')
universe_mask = np.load(config.universe_mask_path, mmap_mode='r')
date_list = np.load(config.date_list_path, allow_pickle=False)
stock_list = np.load(config.stock_list_path, allow_pickle=False)
saved_metadata = json.loads(config.metadata_path.read_text(encoding='utf-8'))

assert tensor.shape == (len(date_list), 6, len(stock_list))
assert valid_mask.shape == universe_mask.shape == (len(date_list), len(stock_list))
assert saved_metadata['feature_order'] == ['open', 'high', 'low', 'close', 'vwap', 'volume']
print('数据准备完成:', tensor.shape)
print('日期:', date_list[0], '至', date_list[-1])
print('特征有效率:', float(valid_mask.mean()))
print('股票池资格率:', float(universe_mask.mean()))